In [1]:
import os
import glob
import pickle
import cv2
import numpy as np

# Define global settings and target file paths
ORB_FEATURES = 500
TRAIN_FOLDER = "../data/train"     # Path to your 200 manipulated fakes
MAP_FILE = "../build/image_map.pkl"

print("Loading database structures...")
with open(MAP_FILE, "rb") as f:
    db_data = pickle.load(f)
    
image_id_map = db_data["image_id_map"]
image_filenames = db_data["image_filenames"]
global_descriptor_matrix = db_data["global_descriptor_matrix"]

# Ensure descriptors are strictly 8-bit integers (required for ORB matching)
if global_descriptor_matrix.dtype != np.uint8:
    global_descriptor_matrix = global_descriptor_matrix.astype(np.uint8)

orb = cv2.ORB_create(nfeatures=ORB_FEATURES)

# Use Brute Force (BFMatcher) instead of LSH for calibration to ensure 100% mathematical accuracy
bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False)

# List to store the performance data of every tested image
results = []

print("\n🚀 Starting batch evaluation of the 200 manipulated images...")

# Locate all valid images in the training folder
image_paths = glob.glob(os.path.join(TRAIN_FOLDER, "*.*"))
valid_extensions = ('.png', '.jpg', '.jpeg')
image_paths = [p for p in image_paths if p.lower().endswith(valid_extensions)]

if not image_paths:
    print(f"❌ ERROR: No images found in '{TRAIN_FOLDER}'. Did you upload them there?")
else:
    for idx, path in enumerate(image_paths):
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if img is None: 
            continue
        
        # Extract features from the manipulated copy
        _, query_descriptors = orb.detectAndCompute(img, None)
        if query_descriptors is None: 
            continue
        
        # --- STAGE 1: Voting via Brute Force Matcher ---
        # Match the fake's features against the entire offline database
        matches = bf.knnMatch(query_descriptors, global_descriptor_matrix, k=1)
        
        votes = np.zeros(len(image_filenames), dtype=np.int32)
        HAMMING_THRESHOLD = 50
        
        for match_list in matches:
            if match_list:
                m = match_list[0]
                if m.distance < HAMMING_THRESHOLD:
                    global_idx = m.trainIdx
                    votes[image_id_map[global_idx]] += 1
                
        winner_id = np.argmax(votes)
        max_votes = votes[winner_id]
        suspect_filename = image_filenames[winner_id]
        
        # --- STAGE 2: Targeted RANSAC Verification ---
        suspect_path = os.path.join("../data/raw", suspect_filename)
        suspect_img = cv2.imread(suspect_path, cv2.IMREAD_GRAYSCALE)
        
        inlier_matches = 0
        if suspect_img is not None:
            kp_suspect, des_suspect = orb.detectAndCompute(suspect_img, None)
            kp_query, des_query = orb.detectAndCompute(img, None)
            
            if des_query is not None and des_suspect is not None:
                # Use BFMatcher with crossCheck for strict 1-to-1 RANSAC pairing
                bf_cc = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
                matches_ransac = bf_cc.match(des_query, des_suspect)
                
                if matches_ransac:
                    src_pts = np.float32([kp_query[m.queryIdx].pt for m in matches_ransac]).reshape(-1, 1, 2)
                    dst_pts = np.float32([kp_suspect[m.trainIdx].pt for m in matches_ransac]).reshape(-1, 1, 2)
                    
                    if len(src_pts) >= 4:
                        M, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 5.0)
                        if mask is not None:
                            inlier_matches = int(np.sum(mask))
                        
        # Record the final scores for this specific fake
        results.append({
            "filename": os.path.basename(path),
            "max_votes": int(max_votes),
            "ransac_inliers": int(inlier_matches)
        })
        
        # Print a status update every 20 images
        if (idx + 1) % 20 == 0 or (idx + 1) == len(image_paths):
            print(f"📊 Processed {idx + 1}/{len(image_paths)} images...")

    # --- 3. Final Aggregation Analysis ---
    if results:
        all_votes = [r["max_votes"] for r in results]
        all_ransac = [r["ransac_inliers"] for r in results]

        print("\n==================================================")
        print("🎯 CALIBRATION ANALYSIS FOR MANIPULATED IMAGES")
        print("==================================================")
        print(f"Total Images Evaluated: {len(results)}")
        
        print("\n[LSH Vote Distribution]")
        print(f"  • Worst-performing fake (MIN votes): {min(all_votes)}")
        print(f"  • Average-performing fake (MEAN votes): {np.mean(all_votes):.1f}")
        print(f"  • Best-performing fake (MAX votes): {max(all_votes)}")
        
        print("\n[RANSAC Inlier Distribution]")
        print(f"  • Lowest geometric match (MIN inliers): {min(all_ransac)}")
        print(f"  • Average geometric match (MEAN inliers): {np.mean(all_ransac):.1f}")
        print(f"  • Highest geometric match (MAX inliers): {max(all_ransac)}")
        print("==================================================")

Loading database structures...

🚀 Starting batch evaluation of the 200 manipulated images...
📊 Processed 20/202 images...
📊 Processed 40/202 images...
📊 Processed 60/202 images...
📊 Processed 80/202 images...
📊 Processed 100/202 images...
📊 Processed 120/202 images...
📊 Processed 140/202 images...
📊 Processed 160/202 images...
📊 Processed 180/202 images...
📊 Processed 200/202 images...
📊 Processed 202/202 images...

🎯 CALIBRATION ANALYSIS FOR MANIPULATED IMAGES
Total Images Evaluated: 202

[LSH Vote Distribution]
  • Worst-performing fake (MIN votes): 8
  • Average-performing fake (MEAN votes): 109.4
  • Best-performing fake (MAX votes): 318

[RANSAC Inlier Distribution]
  • Lowest geometric match (MIN inliers): 19
  • Average geometric match (MEAN inliers): 174.8
  • Highest geometric match (MAX inliers): 347
